# OpenMontage Drama Studio → ComfyUI on Colab

手册：[`docs/COMFY_COLAB.md`](https://github.com/charliewwdev/OpenMontage/blob/main/docs/COMFY_COLAB.md)

**每次：** Runtime → GPU → 顺序跑下面格子 → 把打印的 `COMFY_URL` 填进 Studio `COMFYUI_SERVER_URL`。

⚠️ 免费 **T4(~16GB)** 跑不动完整 MiniMax H3；H3 请用 **Colab Pro 高显存** 或租 4090。T4 仍可用于 SDXL 定妆图联调 API。

## 1) 配置（按需改）

In [ ]:
# 挂载 Drive 后模型可跨会话复用（强烈建议）
USE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/OpenMontageComfy"

# 是否下载 MiniMax H3（约 40GB+，仅 GPU 显存充足时打开）
DOWNLOAD_H3 = False

PORT = 8188

## 2) 挂载 Drive + 安装 ComfyUI

In [ ]:
import os, subprocess, sys
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    WORK = Path(DRIVE_ROOT)
else:
    WORK = Path("/content/OpenMontageComfy")

WORK.mkdir(parents=True, exist_ok=True)
COMFY = WORK / "ComfyUI"
os.chdir(WORK)
print("WORK=", WORK)

if not (COMFY / "main.py").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY)])
else:
    print("ComfyUI already present")

os.chdir(COMFY)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("deps ok")

!nvidia-smi -L || true

## 3) （可选）下载 MiniMax H3 权重

仅当 `DOWNLOAD_H3=True` 且显存够用。文件来自 [Comfy-Org/MiniMax-H3](https://huggingface.co/Comfy-Org/MiniMax-H3)。

In [ ]:
import subprocess
from pathlib import Path

jobs = [
    ("diffusion_models", "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
     "minimax_h3_fl2va_pruned_int8_convrot.safetensors"),
    ("text_encoders", "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
     "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("vae", "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors",
     "minimax_h3_video_vae_fp16.safetensors"),
    ("vae", "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors",
     "minimax_h3_audio_vae_fp32.safetensors"),
]

if not DOWNLOAD_H3:
    print("SKIP H3 download (set DOWNLOAD_H3=True to enable)")
else:
    root = Path("models")
    for folder, url, name in jobs:
        dest = root / folder
        dest.mkdir(parents=True, exist_ok=True)
        path = dest / name
        if path.exists() and path.stat().st_size > 1_000_000:
            print("HAVE", path, path.stat().st_size)
            continue
        print("GET", name)
        subprocess.check_call(["curl", "-L", "--retry", "5", "-C", "-", "-o", str(path), url])
    print("H3 files ready")

## 4) 启动 ComfyUI + cloudflared

跑完后复制输出里的 **`COMFY_URL=https://….trycloudflare.com`** → Drama Studio「ComfyUI 服务器地址」。

In [ ]:
import os, re, socket, subprocess, sys, threading, time
from pathlib import Path

os.chdir(COMFY)

# install cloudflared once
if not Path("/usr/local/bin/cloudflared").exists() and not Path("/usr/bin/cloudflared").exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
    !dpkg -i /tmp/cloudflared.deb >/dev/null

# stop leftover
subprocess.call(["pkill", "-f", "python.*main.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.call(["pkill", "-f", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

comfy = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(PORT), "--enable-cors-header"],
    cwd=str(COMFY),
)
print("ComfyUI pid", comfy.pid)

def wait_port(port, timeout=300):
    t0 = time.time()
    while time.time() - t0 < timeout:
        s = socket.socket(); s.settimeout(1)
        try:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                s.close(); return True
        except Exception:
            pass
        finally:
            s.close()
        time.sleep(1)
    return False

assert wait_port(PORT), "ComfyUI port not open"
print("port", PORT, "open")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

url = None
for line in tunnel.stdout:
    print(line, end="")
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

if not url:
    raise SystemExit("no trycloudflare URL — re-run this cell")

print("\n======== OpenMontage ========")
print("COMFY_URL=" + url)
print("Paste into Drama Studio → COMFYUI_SERVER_URL")
print("Verify:  curl -sS", url + "/system_stats")
print("Keep this cell running. Disconnect runtime when done.")
print("============================\n")

# keep alive
try:
    while True:
        time.sleep(60)
        if comfy.poll() is not None:
            print("ComfyUI exited", comfy.returncode); break
except KeyboardInterrupt:
    print("stopping…")
    comfy.terminate(); tunnel.terminate()

## 5) 接到 Studio（本机执行，不必在 Colab）

```bash
# 自检
curl -sS "$COMFY_URL/system_stats" | head

# 写入本地 studio/.env 后推线上（示例）
# COMFYUI_SERVER_URL=https://xxxx.trycloudflare.com
# COMFYUI_IMAGE_SERVER_URL=…
# COMFYUI_VIDEO_SERVER_URL=…
# STUDIO_URL=https://openmontage.dtok.io ./scripts/restore_studio_keys.sh
```

或在 https://openmontage.dtok.io 模型配置里粘贴同一 URL 并点检测。

**收工：** Colab → Disconnect and delete runtime；清空失效的 `COMFYUI_*_SERVER_URL`。